In [1]:
import sys
from pathlib import Path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException
import selenium
print("selenium版本：", selenium.__version__)

# 建立后面三个阶段共用的保存目录。
PRIVATE = Path.cwd() / "private_data"
PRIVATE.mkdir(exist_ok=True)

# course_visual属于已安装的visual环境，不需要另发文件。
sys.path.insert(0, str(Path.cwd().parent / "L07_L08_visual"))
import course_visual
course_visual.prepare()

# 连接课程Docker中的浏览器和配套驱动。
options = webdriver.ChromeOptions()
options.binary_location = str(next(
    Path("/ms-playwright").glob("chromium-*/chrome-linux*/chrome")
))
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1200,800")
driver_path = Path.home() / ".cache/selenium/chromedriver/linux64/151.0.7922.138/chromedriver"
wd = webdriver.Chrome(service=Service(str(driver_path)), options=options)
# 页面更新时，旧元素可能失效。等待过程中遇到此异常，就重新尝试。
wait = WebDriverWait(wd, 20, ignored_exceptions=(StaleElementReferenceException,))


selenium版本： 4.27.1


### 单元2：打开邮箱，扫码并等待

- 收件箱入口：左侧边栏菜单的文字“收件箱”，用 XPath 按完整文字定位。
- 等待条件：`element_to_be_clickable`（元素可点击），最长等 600 秒（10 分钟），扫码后立即继续。

In [2]:
# ① 打开网页。
wd.get("https://mail.qq.com/")
print("请在浏览器画面中扫码", flush=True)

# ② 本人扫码。登录后页面渲染出左侧边栏，定位“收件箱”菜单项。
inbox = (By.XPATH, '//span[text()="收件箱"]')

# ③ 最长等待600秒，收件箱菜单可点击后立即继续。
WebDriverWait(wd, 600).until(EC.element_to_be_clickable(inbox))
print("登录成功", flush=True)

# ④ 点击进入收件箱。
wd.find_element(*inbox).click()

请在浏览器画面中扫码
登录成功


In [13]:
ROW_CSS = ".mail-list-page-item"      # 一条邮件项目（带data-mailid）
SENDER_CSS = ".cmp-account-nick"      # 发件人姓名
AD_CSS = ".gg-unread-count"           # 广告聚合入口标记
SUBJECT_CSS = ".mail-subject"         # 行内主题
TIME_CSS = ".mail-time"               # 行内时间
HOVER_CSS = ".cmp-account-nick"   # 发件人名字本身（实测：悬停容器中心落在图标上不触发卡片）            # 行内发件人悬停区域
EMAIL_CSS = ".cmp-email-address"      # 页面中邮箱卡片上的邮箱字段（新版QQ邮箱实测类名）

### 单元4：定义列表解析函数

要点：用 `find_elements` 判断广告（返回非空列表即为广告）；读文字用 `.text`；鼠标悬停 `move_to_element(...).perform()`；卡片先等消失再等出现。

In [19]:
import time


# 单元3补丁版：get_ids 加固（跳过渲染中的临时行 + 元素过期自动重读）
def get_ids():
    # 整批ID都在等待函数内读取。重渲染时可能出现没有ID的临时行或旧元素失效：
    # 跳过无ID的行，遇到元素失效就重读整批，直到读到稳定的一批ID。
    def read_current_ids(d):
        for _attempt in range(3):
            try:
                rows = d.find_elements(By.CSS_SELECTOR, ROW_CSS)
                if not rows:
                    return False  # 列表还没出现，继续等待。
                ids = [row.get_attribute("data-mailid") for row in rows]
                ids = [i for i in ids if i]  # 渲染中的临时行没有ID，跳过
                if len(ids) < 2:
                    return False  # 列表还没稳定，继续等待。
                return ids
            except StaleElementReferenceException:
                time.sleep(0.5)  # 旧元素失效，重读整批。
        return False
    return wait.until(read_current_ids)


def get_row(mail_id):
    return wd.find_element(
        By.CSS_SELECTOR, f'{ROW_CSS}[data-mailid="{mail_id}"]'
    )


def read_list_mail(mail_id):
    row = get_row(mail_id)
    # find_elements查找多个元素；存在广告标记时跳过。
    if row.find_elements(By.CSS_SELECTOR, AD_CSS):
        return None

    sender = row.find_element(By.CSS_SELECTOR, SENDER_CSS).text
    subject = row.find_element(By.CSS_SELECTOR, SUBJECT_CSS).text
    time_text = row.find_element(By.CSS_SELECTOR, TIME_CSS).text

    # 先移开鼠标，等1秒让界面稳定，再等上一张卡片消失。
    time_node = row.find_element(By.CSS_SELECTOR, TIME_CSS)
    ActionChains(wd).move_to_element(time_node).perform()
    time.sleep(1)
    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, EMAIL_CSS)))

    # 再悬停当前发件人，等1秒后等卡片出现（太快页面会忽略悬停）。
    # 悬停偶尔不触发卡片：失败就把鼠标移开再重试，最多3次。
    from selenium.common.exceptions import TimeoutException as _SeTimeout
    email = None
    for _attempt in range(3):
        row = get_row(mail_id)  # 重渲染可能使旧行失效，每次重新定位
        time_node = row.find_element(By.CSS_SELECTOR, TIME_CSS)
        sender_node = row.find_element(By.CSS_SELECTOR, HOVER_CSS)
        ActionChains(wd).move_to_element(time_node).perform()
        time.sleep(1)
        ActionChains(wd).move_to_element(sender_node).perform()
        time.sleep(1)
        try:
            email = wait.until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, EMAIL_CSS))
            ).text
            break
        except _SeTimeout:
            pass  # 这一次没弹出卡片，移开鼠标后重试
    if email is None:
        raise TimeoutError(f"悬停3次均未能弹出邮箱卡片：{mail_id}")

    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}


### 单元5：采集10封

先用一封调试（把循环范围改成1个ID核对字段），确认无误后跑完整循环。

In [15]:
basic_records = []
for mail_id in get_ids():
    record = read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))
basic_records[0]

成功采集： 10


{'邮件ID': 'ZC0015_brPNdWOMJIYurWoAdeTM510',
 '发件人': 'Steam Team',
 '邮箱': 'noreply@steampowered.com',
 '主题': 'Steam 上有新的登录操作',
 '时间': '9/15'}

### 单元6：用pandas保存和回读

这两个函数在第三、四阶段继续复用。`index=False` 不多写行号列；`utf-8-sig` 便于Excel打开。

In [16]:
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    # dtype=str按文字读取；空字段保留为空字符串。
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
basic_df.head(10)

CSV记录数： 10


,发件人,邮箱,主题,时间
0,Steam Team,noreply@steampowered.com,Steam 上有新的登录操作,9/15
1,Steam 客服,noreply@steampowered.com,您的 Steam 帐户：来自新网页或移动设备的访问,9/15
2,Steam,noreply@steampowered.com,已移除 Steam 令牌手机验证器,9/15
3,Steam Support,noreply@steampowered.com,Steam 帐户恢复,9/15
4,GitHub,noreply@github.com,[GitHub] A first-party GitHub OAuth applicatio...,9/15
5,GitHub,noreply@github.com,🚀 Your GitHub launch code,9/15
6,赛氪,service@email.saikr.com,下周开考！考场开放两天！获奖率65%！专四专八同主办指导赛事最后一场！,9/11
7,CDA网校,edu@cda.cn,《CDA教材一级：商业数据分析》2025版官方教材入口！搞定考纲新变化实战技能，95% 通关...,9/2
8,CDA网校,edu@cda.cn,《CDA教材一级：商业数据分析》2025版官方教材入口！搞定考纲新变化实战技能，95% 通关...,8/25
9,赛氪,service@email.saikr.com,【重磅开赛】全国大学生翻译大赛正式开启报名！开学加分！,8/21


## 第三阶段：滚动采集与ID去重

### 单元7：批次解析和滚动循环

滚动容器是邮件列表区域的滚动体；若取不到可改用备选 `".mail-list-page-items"`。下一屏位置 = `min(top + clientHeight, scrollHeight - clientHeight)`；用“新旧ID列表不同”判断新一批已加载。

In [17]:
import time

SCROLL_CSS = ".mail-list-page .ui-float-scroll-body"   # 邮件列表滚动容器；备选：".mail-list-page-items"


def get_box():
    return wd.find_element(By.CSS_SELECTOR, SCROLL_CSS)



# 单元7补丁版：parse_batch/collect_batches 加防元素过期重试
def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:  # 已采集过的ID直接跳过
            continue
        # 页面重渲染可能让刚取到的元素失效，重试时按ID重新定位。
        record = None
        for _attempt in range(3):
            try:
                record = read_list_mail(mail_id)
                if record is None:
                    break
                if include_details:
                    # read_detail在第四阶段定义；本阶段False，不执行这里。
                    record.update(read_detail(mail_id))
                break
            except StaleElementReferenceException:
                time.sleep(1.5)
        if record is None:
            continue
        result.append(record)
        seen_ids.add(mail_id)
    return result


def collect_batches(include_details=False, max_mails=0):
    # 每次开始这一轮采集，都回到顶部，并新建本轮集合。
    # 页面重渲染可能让收件箱元素短暂失效，失败就重试。
    for _attempt in range(3):
        try:
            wd.find_element(*inbox).click()
            break
        except StaleElementReferenceException:
            time.sleep(1)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ROW_CSS)))
    wd.execute_script("arguments[0].scrollTop = 0", get_box())
    records, seen_ids = [], set()

    for batch in range(3):  # 本实验共3批
        top = wd.execute_script("return arguments[0].scrollTop", get_box())
        result = parse_batch(get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if batch == 2 or (max_mails and len(seen_ids) >= max_mails):
            break

        # 定位和悬停可能自动滚动，所以仍按本批开始位置计算。
        # 采集一批后页面会重渲染，滚动块整体带重试，box每次重新获取。
        reached_end = False
        for _attempt in range(3):
            try:
                box = get_box()
                height = wd.execute_script("return arguments[0].clientHeight", box)
                full_height = wd.execute_script("return arguments[0].scrollHeight", box)
                next_top = min(top + height, full_height - height)
                if next_top <= top:
                    reached_end = True
                    break
                old_ids = get_ids()
                wd.execute_script("arguments[0].scrollTop = arguments[1]", box, next_top)
                break
            except StaleElementReferenceException:
                time.sleep(1.5)
        if reached_end:
            print("已到列表末尾")
            break
        wait.until(lambda d: get_ids() != old_ids)  # 等待列表内容改变
    return records, seen_ids


### 单元8：执行3批，复用pandas保存函数

In [20]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())

第1批，新增19，累计19
第2批，新增13，累计32
第3批，新增14，累计46
CSV记录数： 46
唯一ID数： 46
重复ID数： 0


### 单元9：重放已采集的ID，验证去重

在**同一次运行**中用同一个 `seen_ids` 重放前5个ID：新增应为0，CSV字节内容不变。

In [21]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"
before = page_path.read_bytes()
replayed = parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))
save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
print("CSV内容未变：", before == page_path.read_bytes())

重复采集新增： 0
CSV内容未变： True


## 第四阶段：详情正文

### 单元10：补全正文处理函数

- 点击行内主题进入详情；正文容器 `.mail-detail-content`；
- `finally` 中点击“返回”回到列表，恢复滚动位置 `top`，并等待当前邮件行重新出现。

In [22]:
import time

BODY_CSS = ".mail-detail-content"
BACK = (By.XPATH, '//*[text()="返回"]')   # 详情页返回控件；若失效请F12核对按钮文字
DOWNLOAD_ATTACHMENTS = False  # 必做先保持False；选做补全后才开启。


def scroll_to_mail(mail_id, timeout=20):
    """QQ邮箱列表为虚拟渲染：滚动到目标邮件行出现在DOM中，才能定位操作。"""
    end = time.time() + timeout
    while time.time() < end:
        try:
            rows = wd.find_elements(By.CSS_SELECTOR, f'{ROW_CSS}[data-mailid="{mail_id}"]')
            if rows:
                return rows[0]
        except StaleElementReferenceException:
            time.sleep(0.5)
            continue
        box = get_box()
        top = wd.execute_script("return arguments[0].scrollTop", box)
        full = wd.execute_script("return arguments[0].scrollHeight", box)
        h = wd.execute_script("return arguments[0].clientHeight", box)
        nxt = min(top + h - 80, full - h)
        wd.execute_script("arguments[0].scrollTop = arguments[1]", box, nxt)
        if nxt <= top:  # 已到底则回到顶部重找
            wd.execute_script("arguments[0].scrollTop = 0", box)
        time.sleep(0.7)
    raise NoSuchElementException(f"滚动后仍找不到 {mail_id}")


def read_detail(mail_id):
    scroll_to_mail(mail_id)  # 目标行可能不在当前视口（虚拟渲染），先滚动定位
    time.sleep(0.8)
    top = wd.execute_script("return arguments[0].scrollTop", get_box())
    row = get_row(mail_id)
    row.find_element(By.CSS_SELECTOR, SUBJECT_CSS).click()
    try:
        body_node = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, BODY_CSS)))
        body = body_node.get_attribute("innerText").strip()
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            # 可选函数在单元13定义，关闭时不会调用。
            files = download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        # 无论正文处理是否成功，都尝试返回列表。
        wd.find_element(*BACK).click()
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, SCROLL_CSS)))
        wd.execute_script("arguments[0].scrollTop = arguments[1]", get_box(), top)
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, f'{ROW_CSS}[data-mailid="{mail_id}"]')
        ))


### 单元11：先试3封

`include_details=True` 开启正文，`max_mails=3` 限制3封。核对3封正文和返回操作后再继续。

In [23]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]

# 先验证3封：进入详情、读正文、返回后继续。
trial_records, _ = collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))

第1批，新增3，累计3
                             邮件ID  \
0  ZC0029_hVjNuqyMbswurWoA80Nwy10   
1  ZC0027_IfzN2syMbM4urWoAsl6ZJ10   
2  ZC0027_ZLnNpbOMbM4urWoA_D3Qc10   

                                                  正文  正文长度  
0  MEMBER EXCLUSIVE\n您的CDA会员专属权益课程已更新\n\n\n\n快来看看...   472  
1  您好，\n\n\n\n\n您的 Oracle 帐户一次性验证码是：\n\n699815\n\...   275  
2  您好，\n\n\n\n\n您的 Oracle 帐户一次性验证码是：\n\n178892\n\...   275  


### 单元12：正式采集3批并保存

In [24]:
detail_records, _ = collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())

第1批，新增19，累计19
第2批，新增13，累计32
第3批，新增14，累计46
正文CSV记录数： 46
重复ID数： 0


### 单元13：选做——附件下载框架（未选做可跳过）

保持 `DOWNLOAD_ATTACHMENTS=False` 时本单元不生效。选做时补全后再运行末尾的3封试采。

In [ ]:
ATTACH_CSS = ".mail-detail-attach-card"          # 附件卡片
DOWNLOAD = (By.XPATH, './/*[text()="下载"]')     # 卡片内“下载”控件；若失效请F12核对


def download_attachments(mail_id):
    cards = wd.find_elements(By.CSS_SELECTOR, ATTACH_CSS)
    if not cards:
        return ""
    folder = PRIVATE / "attachments" / mail_id
    folder.mkdir(parents=True, exist_ok=True)
    wd.execute_cdp_cmd("Browser.setDownloadBehavior", {
        "behavior": "allow", "downloadPath": str(folder.resolve())
    })
    files = []
    for card in cards:
        before = set(folder.iterdir())
        card.find_element(*DOWNLOAD).click()

        def completed(d):
            added = set(folder.iterdir()) - before
            if added and all(p.is_file() and p.suffix != ".crdownload"
                             and p.stat().st_size > 0 for p in added):
                return added
            return False

        added = WebDriverWait(wd, 90).until(completed)
        files.extend(str(p.relative_to(PRIVATE)) for p in added)
    return "\n".join(files)


# 选做验证时再打开下面两行注释；只试3封，不会覆盖正式正文CSV。
# DOWNLOAD_ATTACHMENTS = True
# attachment_records, _ = collect_batches(include_details=True, max_mails=3)
# attachment_path = save_records(attachment_records, "mail_attachments_trial.csv", DETAIL_COLUMNS)
# print(read_records(attachment_path)[["邮件ID", "附件文件"]])
# DOWNLOAD_ATTACHMENTS = False

## 收尾

下课前：保存Notebook → 运行下面单元 → 关闭 `stop-course-visual.bat`。

In [ ]:
# wd.quit()